# Práctica de Procesamiento de Lenguaje Natural (PLN)
## Clasificación de textos con Redes Recurrentes — Elman RNN

---

### ¿Qué vamos a hacer en esta práctica?

Construiremos **desde cero** un clasificador binario de texto basado en una **Red Neuronal Recurrente de Elman (1990)**.  
El modelo leerá una canción entera token a token y decidirá a qué clase pertenece.



---


### Situación en el mapa de modelos vistos

| | Bengio generativo | Bengio clasificador | Elman RNN generativo | **Elman RNN clasificador** |
|---|---|---|---|---|
| Tarea | Predecir siguiente palabra | Clasificación binaria | Predecir siguiente palabra | **Clasificación binaria** |
| Ventana | Fija (`context_size`) | Fija (`context_size`) | Variable (toda la secuencia) | **Variable (toda la secuencia)** |
| Memoria | Ninguna | Ninguna | Estado oculto recurrente | **Estado oculto recurrente** |
| Salida | `vocab_size` logits | 1 logit | `vocab_size` logits por t | **1 logit (estado final h_T)** |

### Idea clave: usar el estado oculto final como representación

En el modelo generativo la RNN produce un logit **en cada paso de tiempo**.  
En el clasificador solo nos interesa el estado oculto **al final de la secuencia** `h_T`:

```
tok₀ → RNN → h₁
tok₁ → RNN → h₂
tok₂ → RNN → h₃
  ...
tok_T → RNN → h_T  ──►  fc  ──►  1 logit  ──►  sigmoid  ──►  clase 0 ó 1
```

`h_T` es un resumen de toda la secuencia: contiene (en teoría) información  
sobre todos los tokens que la RNN ha procesado.

---


## PASO 0: Instalación e Importaciones


In [13]:
# En Google Colab PyTorch ya está disponible.
# Si trabajas en local y no lo tienes, descomenta la siguiente línea:
# !pip install torch

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import re
import random

print(f'PyTorch versión: {torch.__version__}')
print(f'GPU disponible:  {torch.cuda.is_available()}')
print('OK - Librerías importadas correctamente.')


PyTorch versión: 2.11.0+cu130
GPU disponible:  True
OK - Librerías importadas correctamente.


---
## PASO 1: Lectura y Tokenización del Corpus

### Reutilizamos `read_and_tokenize` de las prácticas anteriores

El formato del corpus es idéntico en todas las prácticas: canciones delimitadas por `<CS>...</CS>`,  
con `<EOL>` al final de cada verso. La función no cambia en absoluto.

```
<CS>
Ya te dije frases hechas y deshechas <EOL>
Ya he soñado algún momento de tu piel <EOL>
</CS>
```

> **Hay dos corpus**: uno por clase. El modelo aprenderá a distinguir  
> entre letras del corpus 0 y letras del corpus 1.


In [14]:
import re

def read_and_tokenize(filename):
    """
    Lee un corpus de canciones con formato <CS>...</CS> y lo tokeniza.

    Tokens especiales preservados: <CS>, </CS>, <EOL>
    El resto de puntuación se separa en tokens individuales.
    """
    with open(filename, 'r', encoding='utf-8-sig') as f:
        content = f.read()

    # Patrón que captura: <CS>, </CS>, <EOL>, palabras, y puntuación individual
    tokens = re.findall(r'<CS>|</CS>|<EOL>|\w+|[^\w\s<>/]', content)

    # Agrupamos por canciones: iniciamos nueva lista cada vez que vemos <CS>
    sentences = []
    current_song = []

    for token in tokens:
        if token == '<CS>':
            # Si hay canción en curso, la guardamos (por si hay CS anidados o sueltos)
            if current_song:
                sentences.append(current_song)
            current_song = []
        elif token == '</CS>':
            if current_song:
                sentences.append(current_song)
            current_song = []
        else:
            current_song.append(token)

    # Por si queda algo sin cerrar
    if current_song:
        sentences.append(current_song)

    return sentences

# Carga del corpus
Actualiza las variables path_corpus_0 y path_corpus_1 con tus corpus de trabajo



In [15]:
path_corpus_0 = 'corpusA.txt'
sentences_class0 = read_and_tokenize(path_corpus_0)

print(f'Lectura y tokenización del corpus 0 finalizada.')
print(f"🔢 Total de sentencias en el corpus 0: {len(sentences_class0)}")

path_corpus_1 = 'corpusB.txt'
sentences_class1 = read_and_tokenize(path_corpus_1)

print(f'Lectura y tokenización del corpus 1 finalizada.')
print(f"🔢 Total de sentencias en el corpus 1: {len(sentences_class1)}")


MAX_CANCIONES = 5000
sentences_class0 = sentences_class0[:MAX_CANCIONES]  # Primeras canciones para agilizar
sentences_class1 = sentences_class1[:MAX_CANCIONES]  # Primeras canciones para agilizar


print(f'Canciones clase 0: {len(sentences_class0)}')
print(f'Canciones clase 1: {len(sentences_class1)}')
print(f'\nEjemplo clase 0 (primeros 8 tokens): {sentences_class0[0][:8]}')
print(f'Ejemplo clase 1 (primeros 8 tokens): {sentences_class1[0][:8]}')

Lectura y tokenización del corpus 0 finalizada.
🔢 Total de sentencias en el corpus 0: 6606
Lectura y tokenización del corpus 1 finalizada.
🔢 Total de sentencias en el corpus 1: 1542
Canciones clase 0: 5000
Canciones clase 1: 1542

Ejemplo clase 0 (primeros 8 tokens): ['Tú', 'y', 'yo', 'en', 'mi', 'habitación', '<EOL>', 'Y']
Ejemplo clase 1 (primeros 8 tokens): ['Ya', 'te', 'dije', 'frases', 'hechas', 'y', 'deshechas', '<EOL>']


---
## PASO 2: Construcción del Vocabulario

### Clase `Vocabulary` con filtrado por frecuencia mínima K

Igual que en las prácticas anteriores, el vocabulario es un diccionario bidireccional.  
Se añaden dos mejoras:

- **`<UNK>`** *(índice 1)*: representa tokens de baja frecuencia o no vistos en entrenamiento.
- **`build_from_sentences(sentences, min_freq)`**: construye el vocabulario incluyendo solo
  los tokens con frecuencia `>= min_freq`. El resto queda mapeado a `<UNK>` al codificar.

> **Importante:** el vocabulario se construye **solo** sobre las frases de entrenamiento  
> para evitar *data leakage*. Los tokens del test no vistos en train también reciben `<UNK>`.


In [16]:
from collections import Counter

class Vocabulary:
    """
    Diccionario bidireccional: palabra <-> índice numérico.

    Tokens especiales reservados:
        <PAD>  índice 0 — relleno para longitud fija
        <UNK>  índice 1 — tokens desconocidos o de baja frecuencia
    """

    def __init__(self):
        self.token_to_idx = {}
        self.idx_to_token = {}
        self.size = 0

    def add_token(self, token):
        """Añade un token al vocabulario si no existe ya."""
        if token not in self.token_to_idx:
            self.token_to_idx[token] = self.size
            self.idx_to_token[self.size] = token
            self.size += 1

    def build_from_sentences(self, sentences, min_freq):
        """
        Construye el vocabulario a partir de una lista de frases,
        incluyendo solo los tokens con frecuencia >= min_freq.
        Los tokens con frecuencia < min_freq quedan mapeados a <UNK>.

        Args:
            sentences (list[list[str]]): corpus tokenizado.
            min_freq  (int): frecuencia mínima para entrar en el vocabulario.
        """
        freq = Counter(tok for sentence in sentences for tok in sentence)
        for token, count in freq.items():
            if count >= min_freq:
                self.add_token(token)
        discarded = sum(1 for c in freq.values() if c < min_freq)
        print(f'  Tokens únicos en corpus   : {len(freq)}')
        print(f'  Tokens descartados (< {min_freq}) : {discarded}  -> mapeados a <UNK>')
        print(f'  Tokens en vocabulario      : {self.size}')

    def encode(self, tokens):
        """
        Convierte lista de palabras en lista de índices numéricos.
        Los tokens fuera del vocabulario se mapean al índice de <UNK> (1).
        """
        unk_idx = self.token_to_idx['<UNK>']
        return [self.token_to_idx.get(tok, unk_idx) for tok in tokens]

    def decode(self, indices):
        """Convierte lista de índices en lista de palabras."""
        return [self.idx_to_token[idx] for idx in indices]


# ── Parámetro de frecuencia mínima ──────────────────────────────────────────
MIN_FREQ = 2   # Tokens con frecuencia < MIN_FREQ serán reemplazados por <UNK>

# El vocabulario se construye SOLO sobre frases de entrenamiento.
# Hacemos el split aquí con la misma semilla que en el Paso 3.
_labels_class0 = [0] * len(sentences_class0)
_labels_class1 = [1] * len(sentences_class1)
_combined = list(zip(sentences_class0 + sentences_class1,
                     _labels_class0    + _labels_class1))
random.seed(42)
random.shuffle(_combined)
_sentences_all, _labels_all = zip(*_combined)

TRAIN_RATIO = 0.8
_split_idx  = int(len(_sentences_all) * TRAIN_RATIO)
train_sentences_for_vocab = _sentences_all[:_split_idx]

vocab = Vocabulary()
vocab.add_token('<PAD>')   # índice 0 — relleno
vocab.add_token('<UNK>')   # índice 1 — desconocido / baja frecuencia

print(f'Construyendo vocabulario con MIN_FREQ = {MIN_FREQ}...')
vocab.build_from_sentences(train_sentences_for_vocab, min_freq=MIN_FREQ)

print(f'\nVocabulario construido.')
print(f'  Tamaño total : {vocab.size} tokens')
print(f'  Índice <PAD> : {vocab.token_to_idx["<PAD>"]}')
print(f'  Índice <UNK> : {vocab.token_to_idx["<UNK>"]}')
print(f'  Índice <EOL> : {vocab.token_to_idx.get("<EOL>", "No encontrado")}')

# Ejemplo: token inventado debe aparecer como <UNK>
ejemplo = sentences_class0[0][:3] + ['TOKEN_INVENTADO']
print(f'\nEjemplo encode ("TOKEN_INVENTADO" debe ser <UNK>=1):')
print(f'  Tokens  : {ejemplo}')
print(f'  Índices : {vocab.encode(ejemplo)}')
print(f'  Decode  : {vocab.decode(vocab.encode(ejemplo))}')


Construyendo vocabulario con MIN_FREQ = 2...
  Tokens únicos en corpus   : 49271
  Tokens descartados (< 2) : 20440  -> mapeados a <UNK>
  Tokens en vocabulario      : 28833

Vocabulario construido.
  Tamaño total : 28833 tokens
  Índice <PAD> : 0
  Índice <UNK> : 1
  Índice <EOL> : 9

Ejemplo encode ("TOKEN_INVENTADO" debe ser <UNK>=1):
  Tokens  : ['Tú', 'y', 'yo', 'TOKEN_INVENTADO']
  Índices : [1307, 25, 105, 1]
  Decode  : ['Tú', 'y', 'yo', '<UNK>']


---
## PASO 3: Preparación del Dataset de Clasificación

### Reutilizamos `SentenceDataset` del clasificador Bengio

La clase es **idéntica** a la del clasificador Bengio: aplica padding o truncado  
para que cada canción tenga exactamente `context_size` tokens.

### Manejo de tokens fuera del vocabulario en clasificación
El vocabulario fue construido en el Paso 2 **solo** con las frases de entrenamiento y con `MIN_FREQ`.  
Al codificar cualquier frase (entrenamiento o test), `vocab.encode()` mapea automáticamente  
los tokens no presentes en el vocabulario al índice de **`<UNK>`**.

> **¿Por qué padding/truncado si la RNN maneja longitudes variables?**  
> Técnicamente la RNN puede procesar secuencias de cualquier longitud.  
> Sin embargo, agrupar ejemplos en batches exige tensores de igual forma.  
> El padding hasta `context_size` es la solución más sencilla y directa.  
>
> La alternativa sería usar `PackedSequence` (más eficiente pero más compleja).  
> La veremos como extensión opcional al final.

### Comparativa de representaciones de entrada

| Modelo | Representación de la canción | Memoria del contexto |
|---|---|---|
| Bengio generativo | Ventana de N tokens | Ninguna |
| Bengio clasificador | Secuencia completa (pad/trunca) | Ninguna — suma de embeddings |
| Elman RNN generativo | Secuencia completa | Estado oculto hₜ por cada t |
| **Elman RNN clasificador** | **Secuencia completa (pad/trunca)** | **Estado oculto final h_T** |


In [17]:
class SentenceDataset(Dataset):
    """
    Envuelve pares (secuencia_codificada, etiqueta) en el formato de PyTorch.

    Aplica padding o truncado para garantizar longitud fija = context_size.
    Los tokens no presentes en el vocabulario se mapean a <UNK> durante
    la codificación (gestionado por vocab.encode).
    """

    def __init__(self, sentences, labels, vocab, context_size):
        self.vocab        = vocab
        self.context_size = context_size
        self.data         = []
        self.pad_idx      = vocab.token_to_idx['<PAD>']

        for sentence, label in zip(sentences, labels):
            # vocab.encode mapea tokens OOV o de baja frecuencia a <UNK>
            encoded = vocab.encode(sentence)

            # Padding o truncado hasta context_size
            if len(encoded) < context_size:
                encoded = encoded + [self.pad_idx] * (context_size - len(encoded))
            else:
                encoded = encoded[:context_size]

            self.data.append((encoded, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        inputs, label = self.data[idx]
        return (
            torch.tensor(inputs, dtype=torch.long),
            torch.tensor(label,  dtype=torch.float)
        )


# ── Parámetros del dataset ────────────────────────────────────────────────────
CONTEXT_SIZE = 30

labels_class0 = [0] * len(sentences_class0)
labels_class1 = [1] * len(sentences_class1)

# Mezclamos con la misma semilla usada en el Paso 2
sentences_all = sentences_class0 + sentences_class1
labels_all    = labels_class0    + labels_class1

combined = list(zip(sentences_all, labels_all))
random.seed(42)
random.shuffle(combined)
sentences_all, labels_all = zip(*combined)

# NOTA: usa el mismo TRAIN_RATIO definido en el Paso 2
split_idx = int(len(sentences_all) * TRAIN_RATIO)

train_sentences, test_sentences = sentences_all[:split_idx], sentences_all[split_idx:]
train_labels,    test_labels    = labels_all[:split_idx],    labels_all[split_idx:]

# - train_dataset: tokens < MIN_FREQ ya excluidos del vocab -> codificados como <UNK>
# - test_dataset : tokens OOV (no vistos en train) -> también codificados como <UNK>
train_dataset = SentenceDataset(train_sentences, train_labels, vocab, CONTEXT_SIZE)
test_dataset  = SentenceDataset(test_sentences,  test_labels,  vocab, CONTEXT_SIZE)

print(f'Dataset preparado.')
print(f'  Total ejemplos : {len(sentences_all)}')
print(f'  Entrenamiento  : {len(train_dataset)}  |  Test: {len(test_dataset)}')
print(f'  Clase 0        : {list(labels_all).count(0)}  |  Clase 1: {list(labels_all).count(1)}')
print(f'  context_size   : {CONTEXT_SIZE} tokens por ejemplo')

inputs_ex, label_ex = train_dataset[0]
print(f'\nEjemplo:')
print(f'  Forma tensor entrada : {inputs_ex.shape}')
print(f'  Etiqueta             : {label_ex.item():.0f}')
print(f'  Primeros 6 tokens    : {vocab.decode(inputs_ex[:6].tolist())}')
print(f'  (Los <UNK> indican tokens de baja frecuencia o no vistos en entrenamiento)')


Dataset preparado.
  Total ejemplos : 6542
  Entrenamiento  : 5233  |  Test: 1309
  Clase 0        : 5000  |  Clase 1: 1542
  context_size   : 30 tokens por ejemplo

Ejemplo:
  Forma tensor entrada : torch.Size([30])
  Etiqueta             : 1
  Primeros 6 tokens    : ['Mañana', 'me', 'voy', 'p', "'", 'al']
  (Los <UNK> indican tokens de baja frecuencia o no vistos en entrenamiento)


---
## PASO 4: Definición del Modelo Elman RNN Clasificador

### Arquitectura completa

```
ENTRADA:  [tok₀, tok₁, tok₂, ..., tok_T]   <- context_size índices

              │
    +---------+---------+
    |   EMBEDDING       |  tokₜ (índice) -> eₜ (vector embed_size)
    |   nn.Embedding    |  Procesamos la secuencia entera de una vez
    +---------+---------+
              │
    +---------+---------+
    |   CAPA RNN        |  Procesa e₀, e₁, ..., e_T de izquierda a derecha
    |   nn.RNN          |  hₜ = tanh(Wₓ·eₜ + Wₕ·hₜ₋₁ + b)
    +---------+---------+  Mantiene memoria del contexto pasado
              │
              │  Solo usamos h_T (estado oculto del ÚLTIMO token)
              │  h_T es un resumen de toda la secuencia
              ▼
    +---------+---------+
    |  CAPA DE SALIDA   |  h_T -> 1 logit  (clasificación binaria)
    |  nn.Linear        |
    +---------+---------+
              │
SALIDA:   logit ∈ ℝ  ->  sigmoid(logit) > 0.5  ->  Clase 1, si no -> Clase 0
```

### Diferencia clave respecto al modelo generativo

| Aspecto | Elman RNN generativo | **Elman RNN clasificador** |
|---|---|---|
| Salida de `fc` | `vocab_size` logits por cada posición t | **1 logit usando solo h_T** |
| Estado oculto usado | Todos (h₀...h_T) | **Solo el último (h_T)** |
| Loss | CrossEntropyLoss | **BCEWithLogitsLoss** |
| Objetivo | Predecir siguiente palabra | **Predecir clase 0 ó 1** |

### ¿Por qué `hidden[-1]` y no `output[:, -1, :]`?

`nn.RNN` devuelve dos valores:
- `output`: tensor `[batch, seq_len, hidden_dim]` — estados ocultos en **cada posición**
- `hidden`: tensor `[num_layers, batch, hidden_dim]` — estado oculto de la **última posición**

Para una RNN de una sola capa, `hidden[-1]` == `output[:, -1, :]`.  
Usar `hidden[-1]` es más explícito sobre la intención: queremos el **resumen final**.


In [18]:
class ElmanRNN(nn.Module):
    """
    Red Neuronal Recurrente de Elman adaptada a clasificación binaria.

    Procesa la secuencia de tokens de una canción y predice su clase (0 ó 1)
    usando el estado oculto del último token como representación global.

    Arquitectura:
        Embedding -> nn.RNN -> fc (1 logit) usando hidden[-1]

    Args:
        vocab_size (int):  Número de palabras en el vocabulario.
        embed_size (int):  Dimensión de los vectores de embedding.
        hidden_dim (int):  Dimensión del estado oculto de la RNN.
    """

    def __init__(self, vocab_size, embed_size, hidden_dim):
        super(ElmanRNN, self).__init__()

        # Tabla de embeddings — idéntica a todas las prácticas anteriores
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # Capa RNN de Elman
        # batch_first=True -> espera [batch, seq_len, embed_size]
        self.rnn = nn.RNN(embed_size, hidden_dim, batch_first=True)

        # Capa de salida: UN único logit para clasificación binaria
        # Recibe el estado oculto final h_T (tamaño hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        """
        Propagación hacia adelante.

        Args:
            x : Tensor [batch, seq_len] — índices de tokens.

        Returns:
            logits : Tensor [batch] — un logit por ejemplo.
        """
        # 1. Lookup de embeddings
        #    [batch, seq_len] -> [batch, seq_len, embed_size]
        emb = self.embedding(x)

        # 2. Capa RNN — procesa toda la secuencia
        #    output : [batch, seq_len, hidden_dim]  — todos los estados ocultos
        #    hidden : [1, batch, hidden_dim]         — estado oculto final h_T
        output, hidden = self.rnn(emb)

        # 3. Usamos SOLO el estado oculto final como representación de la canción
        #    hidden[-1] : [batch, hidden_dim]
        #    (para RNN de 1 capa: hidden[-1] == output[:, -1, :])
        final_hidden = hidden[-1]   # [batch, hidden_dim]

        # 4. Capa de salida: 1 logit por ejemplo
        #    [batch, hidden_dim] -> [batch, 1] -> [batch]
        logits = self.fc(final_hidden).squeeze(-1)

        return logits


# ── Hiperparámetros del modelo ────────────────────────────────────────────────
EMBED_SIZE = 128
HIDDEN_DIM = 256

model = ElmanRNN(
    vocab_size = vocab.size,
    embed_size = EMBED_SIZE,
    hidden_dim = HIDDEN_DIM
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Modelo ElmanRNN clasificador creado.')
print(f'\nArquitectura:')
print(model)
print(f'\nParámetros entrenables: {total_params:,}')
print(f'  - Embedding : {vocab.size} × {EMBED_SIZE}  = {vocab.size * EMBED_SIZE:,}')
print(f'  - RNN (Wx+Wh+b): ~{EMBED_SIZE*HIDDEN_DIM + HIDDEN_DIM*HIDDEN_DIM + HIDDEN_DIM:,}')
print(f'  - fc (salida)   : {HIDDEN_DIM} × 1  = {HIDDEN_DIM:,}')

# Test rápido de dimensiones
dummy = torch.zeros(3, CONTEXT_SIZE, dtype=torch.long)   # batch=3
out   = model(dummy)
print(f'\nTest de dimensiones (batch=3, seq_len={CONTEXT_SIZE}):')
print(f'  Salida: {out.shape}   (batch — un logit por ejemplo)')


Modelo ElmanRNN clasificador creado.

Arquitectura:
ElmanRNN(
  (embedding): Embedding(28833, 128)
  (rnn): RNN(128, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)

Parámetros entrenables: 3,789,697
  - Embedding : 28833 × 128  = 3,690,624
  - RNN (Wx+Wh+b): ~98,560
  - fc (salida)   : 256 × 1  = 256

Test de dimensiones (batch=3, seq_len=30):
  Salida: torch.Size([3])   (batch — un logit por ejemplo)


---
## PASO 5: DataLoader de PyTorch

### Reutilizamos el DataLoader del clasificador Bengio

Como `SentenceDataset` ya produce tensores de tamaño fijo (`context_size`),  
**no necesitamos `collate_fn` personalizada**: PyTorch puede apilarlos directamente.

> **Comparativa con el Elman RNN generativo:**  
> En el modelo generativo las canciones tienen longitud variable → necesitábamos  
> `rnn_collate_fn` con `pad_sequence`. Aquí el padding ya está en el dataset,  
> así que el `DataLoader` estándar funciona sin modificaciones.


In [19]:
BATCH_SIZE = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'DataLoaders listos.')
print(f'  Train — ejemplos: {len(train_dataset)}  |  batches: {len(train_loader)}')
print(f'  Test  — ejemplos: {len(test_dataset)}   |  batches: {len(test_loader)}')

sample_inputs, sample_labels = next(iter(train_loader))
print(f'\nForma tensor entradas : {sample_inputs.shape}   (batch × context_size)')
print(f'Forma tensor etiquetas: {sample_labels.shape}   (batch)')
print(f'Etiquetas del batch   : {sample_labels.tolist()}')


DataLoaders listos.
  Train — ejemplos: 5233  |  batches: 655
  Test  — ejemplos: 1309   |  batches: 164

Forma tensor entradas : torch.Size([8, 30])   (batch × context_size)
Forma tensor etiquetas: torch.Size([8])   (batch)
Etiquetas del batch   : [1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0]


---
## PASO 6: Entrenamiento del Modelo

### Reutilizamos el ciclo de entrenamiento del clasificador Bengio

El ciclo es **idéntico** al del clasificador Bengio: misma loss (`BCEWithLogitsLoss`),  
mismo optimizador (`Adam`), mismo esquema de 4 pasos.

La única diferencia es que ahora el forward pass pasa por la RNN en lugar de por  
una suma de embeddings — pero eso es interno al modelo; el bucle de entrenamiento no cambia.

```
PASO 1 — FORWARD: la RNN procesa la secuencia y devuelve 1 logit (h_T → fc)
PASO 2 — LOSS:    BCEWithLogitsLoss compara el logit con la etiqueta real
PASO 3 — BACKWARD: gradientes retropropagados a través del tiempo (BPTT)
PASO 4 — UPDATE:  Adam actualiza Wₓ, Wₕ, embeddings y fc
```

### BPTT — Backpropagation Through Time

El gradiente de la loss respecto a los pesos de la RNN se calcula  
**hacia atrás en el tiempo** desde el último token hasta el primero.  
Esto permite que el modelo aprenda dependencias a largo plazo,  
aunque con el riesgo del **gradiente evanescente** en secuencias muy largas.


In [20]:
def train_model(model, data_loader, num_epochs, learning_rate, device='cpu'):
    """
    Entrena el clasificador binario ElmanRNN.

    Reutilizada del clasificador Bengio con mínimos cambios:
    - El forward ya devuelve un logit escalar (sin squeeze adicional necesario).
    - El ciclo 4-pasos es idéntico.

    Args:
        model         : Modelo ElmanRNN.
        data_loader   : DataLoader de entrenamiento.
        num_epochs    : Número de épocas.
        learning_rate : Tasa de aprendizaje para Adam.
        device        : 'cpu' o 'cuda'.
    """
    model.to(device)
    model.train()

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    total_batches = len(data_loader)
    print(f'Iniciando entrenamiento en {device}')
    print(f'  Épocas: {num_epochs}  |  LR: {learning_rate}  |  Batch: {data_loader.batch_size}')
    print('-' * 65)

    for epoch in range(num_epochs):
        total_loss = 0.0

        for inputs, labels in data_loader:
            inputs = inputs.to(device)           # [batch, context_size]
            labels = labels.to(device).float()   # [batch]

            # ── PASO 1: Forward pass ──────────────────────────────────────
            outputs = model(inputs)              # [batch]

            # ── PASO 2: Cálculo de la pérdida ────────────────────────────
            loss = criterion(outputs, labels)

            # ── PASO 3: Backward pass (BPTT) ─────────────────────────────
            optimizer.zero_grad()
            loss.backward()

            # ── PASO 4: Actualización de pesos ────────────────────────────
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / total_batches
        print(f'Época [{epoch+1:3d}/{num_epochs}]  |  Loss: {avg_loss:.4f}')

    print('-' * 65)
    print('Entrenamiento completado.')


NUM_EPOCHS    = 10
LEARNING_RATE = 0.0001
device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_model(
    model         = model,
    data_loader   = train_loader,
    num_epochs    = NUM_EPOCHS,
    learning_rate = LEARNING_RATE,
    device        = device
)


Iniciando entrenamiento en cuda
  Épocas: 10  |  LR: 0.0001  |  Batch: 8
-----------------------------------------------------------------
Época [  1/10]  |  Loss: 0.5700
Época [  2/10]  |  Loss: 0.5377
Época [  3/10]  |  Loss: 0.5216
Época [  4/10]  |  Loss: 0.5032
Época [  5/10]  |  Loss: 0.4763
Época [  6/10]  |  Loss: 0.4487
Época [  7/10]  |  Loss: 0.4217
Época [  8/10]  |  Loss: 0.3937
Época [  9/10]  |  Loss: 0.3618
Época [ 10/10]  |  Loss: 0.3329
-----------------------------------------------------------------
Entrenamiento completado.


---
## PASO 7: Evaluación del Modelo

### Reutilizamos `evaluate_model` del clasificador Bengio

La función de evaluación es **idéntica**: mismo umbral 0.5, misma métrica accuracy.

```
prediccion = (sigmoid(logit) > 0.5) → 0 ó 1
accuracy   = predicciones_correctas / total_ejemplos
```

> **`model.eval()` y `torch.no_grad()`** son igual de importantes que en Bengio:  
> desactivan comportamientos de entrenamiento y ahorran memoria.


In [21]:
def evaluate_model(model, data_loader, device='cpu'):
    """
    Evalúa la exactitud del clasificador ElmanRNN.

    Reutilizada del clasificador Bengio — sin cambios.

    Args:
        model       : Modelo ya entrenado.
        data_loader : DataLoader de evaluación.
        device      : 'cpu' o 'cuda'.

    Returns:
        float: Exactitud entre 0 y 1.
    """
    model.to(device)
    model.eval()

    correct = 0
    total   = 0

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs     = model(inputs)                           # [batch]
            predictions = (torch.sigmoid(outputs) > 0.5).long()  # 0 ó 1
            labels_long = labels.long()

            correct += (predictions == labels_long).sum().item()
            total   += labels.size(0)

    return correct / total


train_accuracy = evaluate_model(model, train_loader, device=device)
test_accuracy  = evaluate_model(model, test_loader,  device=device)

print(f'Resultados finales')
print(f'  Exactitud entrenamiento : {train_accuracy:.4f}  ({train_accuracy*100:.1f}%)')
print(f'  Exactitud test          : {test_accuracy:.4f}  ({test_accuracy*100:.1f}%)')

if train_accuracy - test_accuracy > 0.15:
    print('\n⚠️  Posible sobreajuste (overfitting):')
    print('   Prueba a reducir NUM_EPOCHS, HIDDEN_DIM o aumentar el corpus.')
elif test_accuracy >= 0.7:
    print('\n✓  El modelo generaliza razonablemente bien.')
else:
    print('\n→  Prueba a aumentar NUM_EPOCHS o el tamaño del corpus.')


Resultados finales
  Exactitud entrenamiento : 0.8962  (89.6%)
  Exactitud test          : 0.6906  (69.1%)

⚠️  Posible sobreajuste (overfitting):
   Prueba a reducir NUM_EPOCHS, HIDDEN_DIM o aumentar el corpus.


---
## Análisis de Predicciones

Inspeccionamos las predicciones del modelo sobre el conjunto de test.


In [22]:
model.eval()
print(f'{'REAL':>6}  {'PRED':>6}  {'PROB CL.1':>10}  {'OK':>4}  PRIMEROS TOKENS')
print('-' * 72)

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        logits = model(inputs)
        probs  = torch.sigmoid(logits)
        preds  = (probs > 0.5).long()

        for i in range(len(labels)):
            real     = int(labels[i].item())
            pred     = int(preds[i].item())
            prob_c1  = probs[i].item()
            ok       = '✓' if real == pred else '✗'
            primeras = vocab.decode(inputs[i, :6].cpu().tolist())
            print(f'{real:>6}  {pred:>6}  {prob_c1:>10.3f}  {ok:>4}  {" ".join(primeras)}...')


  REAL    PRED   PROB CL.1    OK  PRIMEROS TOKENS
------------------------------------------------------------------------
     0       0       0.189     ✓  Vámonos a alguna isla , <UNK>...
     1       0       0.488     ✗  Doña María le ruego <EOL> En...
     0       1       0.584     ✗  No creas que no me doy...
     0       0       0.213     ✓  Estuve bien estuve mal <EOL> Tuve...
     0       0       0.008     ✓  El derecho de vivir <EOL> Sin...
     0       1       0.596     ✗  Tengo tiempo para saber <EOL> Si...
     1       0       0.112     ✗  Cielo arriba de <UNK> <EOL> Camino...
     0       1       0.884     ✗  Trago <EOL> Me hago el idiota...
     0       0       0.308     ✓  Lo que <UNK> esta noche es...
     1       0       0.116     ✗  <UNK> que vas <UNK> <EOL> <UNK>...
     0       1       0.714     ✗  <UNK> mintiendo y no te <UNK>...
     0       1       0.922     ✗  ¿ Por qué este deseo crece...
     0       0       0.131     ✓  Fluorescente azul <EOL> Luz que baña...

---
## Comparativa: Bengio Clasificador vs. Elman RNN Clasificador

Ambos modelos resuelven el mismo problema (clasificación binaria de canciones)  
pero con estrategias de representación muy distintas. Reflexiona sobre estas diferencias:

### Representación de la secuencia

```
Bengio clasificador:
  [tok₀, tok₁, ..., tok_T]  →  Σ embeddings  →  vector fijo  →  fc  →  logit
                                ─────────────
                                Sin orden: 'perro muerde hombre' == 'hombre muerde perro'

Elman RNN clasificador:
  [tok₀, tok₁, ..., tok_T]  →  h₀→h₁→...→h_T  →  h_T  →  fc  →  logit
                                ──────────────────
                                Con orden: el estado hₜ depende de todo lo anterior
```

### ¿Cuándo la RNN tiene ventaja sobre Bengio?

- Cuando el **orden de las palabras** importa para la clasificación.
- Cuando hay **dependencias largas** (la clase depende de algo dicho al principio).
- Cuando las canciones son **largas** (Bengio trunca y pierde información).


---
## Cuestionario de Reflexión

---

**Pregunta 1:** El clasificador Bengio usa la **suma de embeddings** como representación; el Elman RNN usa **h_T** (el estado oculto final). ¿Qué información tiene h_T que la suma no puede capturar? Pon un ejemplo concreto con una frase corta.

**Tu respuesta:**
*(Escribe tu respuesta aquí...)*

---

**Pregunta 2:** En el forward de este modelo usamos `hidden[-1]` en lugar de `output[:, -1, :]`. Explica qué contiene `output` y qué contiene `hidden` en una RNN de una sola capa. ¿Son siempre iguales estos dos accesos?

**Tu respuesta:**
*(Escribe tu respuesta aquí...)*

---

**Pregunta 3:** El `context_size` en este modelo es 30 tokens pero en el código original era 300. ¿Qué consecuencias tiene un `context_size` muy grande para: (a) el consumo de memoria, (b) la capacidad de capturar contexto, (c) el gradiente evanescente?

**Tu respuesta:**
*(Escribe tu respuesta aquí...)*

---

**Pregunta 4:** A diferencia del Elman RNN generativo, aquí no necesitamos `collate_fn` personalizada. ¿Por qué? ¿En qué situación sería preferible usar `PackedSequence` en lugar de padding fijo?

**Tu respuesta:**
*(Escribe tu respuesta aquí...)*

---

**Pregunta 5:** Si el dataset tiene 30 canciones de clase 0 y 30 de clase 1 (balanceado), un clasificador que siempre predice la clase mayoritaria obtiene un 50% de accuracy. ¿Qué accuracy mínimo debería alcanzar tu modelo para demostrar que ha aprendido algo útil? ¿Cómo cambiaría esta cota si el dataset fuera 90/10?

**Tu respuesta:**
*(Escribe tu respuesta aquí...)*

---

**Pregunta 6 (Investigación):** Tanto la RNN como el clasificador Bengio son **modelos de bolsa de palabras con embeddings** o **modelos de secuencia**. ¿Qué arquitecturas más modernas (post-2017) han superado a las RNNs en clasificación de texto? ¿Cuál es su ventaja principal?

**Tu respuesta:**
*(Escribe tu respuesta aquí...)*

---


---
## Extensión opcional: Experimenta con los hiperparámetros

| Hiperparámetro | Valor base | Efecto al aumentar |
|---|---|---|
| `CONTEXT_SIZE` | 30 | Más contexto por canción (más memoria GPU) |
| `EMBED_SIZE` | 128 | Representaciones más ricas |
| `HIDDEN_DIM` | 256 | Mayor capacidad de memoria recurrente |
| `NUM_EPOCHS` | 10 | Más entrenamiento (riesgo de sobreajuste) |
| `LEARNING_RATE` | 0.0001 | Convergencia más rápida (riesgo de inestabilidad) |
| `BATCH_SIZE` | 8 | Mayor batch = gradientes más estables |

### Variante avanzada: RNN multicapa

```python
self.rnn = nn.RNN(embed_size, hidden_dim, num_layers=2, batch_first=True)
# hidden[-1] sigue siendo el estado final de la ÚLTIMA capa
```

### Variante avanzada: usar la media de todos los estados ocultos

En lugar de usar solo h_T, podemos promediar todos los estados ocultos:

```python
# En forward(), sustituir hidden[-1] por:
final_hidden = output.mean(dim=1)   # [batch, hidden_dim]
```

¿Mejora la clasificación? ¿Por qué podría ser útil en canciones largas?


In [23]:
# ── ZONA DE EXPERIMENTOS ──────────────────────────────────────────────────────
CONTEXT_SIZE  = 30
EMBED_SIZE    = 128
HIDDEN_DIM    = 256
NUM_EPOCHS    = 10
LEARNING_RATE = 0.0001
BATCH_SIZE    = 8

print('Valores actuales:')
print(f'  CONTEXT_SIZE  = {CONTEXT_SIZE}')
print(f'  EMBED_SIZE    = {EMBED_SIZE}')
print(f'  HIDDEN_DIM    = {HIDDEN_DIM}')
print(f'  NUM_EPOCHS    = {NUM_EPOCHS}')
print(f'  LEARNING_RATE = {LEARNING_RATE}')
print(f'  BATCH_SIZE    = {BATCH_SIZE}')
print('\nVuelve a ejecutar desde el Paso 3 para aplicar los cambios.')


Valores actuales:
  CONTEXT_SIZE  = 30
  EMBED_SIZE    = 128
  HIDDEN_DIM    = 256
  NUM_EPOCHS    = 10
  LEARNING_RATE = 0.0001
  BATCH_SIZE    = 8

Vuelve a ejecutar desde el Paso 3 para aplicar los cambios.
